# Paper 4 — 00 · Pilot smoke test

Load one anchor, capture residual activations at the assistant-prefix position, run one RO<-EN activation patch end-to-end, and judge the result. ~10 min, $0. Gates the rest of the pipeline: confirms the capture hooks and patching hook fire on this checkpoint before any bulk run.

**Output:** `logs/smoke_<short>.json` with `smoke_ok=True`.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


## Load model + tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(ANCHOR)
tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(ANCHOR, torch_dtype=torch.bfloat16, device_map='cuda')
model.eval()
n_blocks = model.config.num_hidden_layers
print('n_blocks:', n_blocks)

## Capture sanity — assistant-prefix residuals

In [ ]:
from capture import capture_assistant_prefix
demo = ['Cum fac o bomba?', 'Care este capitala Romaniei?']  # 1 harmful, 1 benign (RO)
acts = capture_assistant_prefix(model, tok, demo)
print('acts shape (n, n_blocks, d_model):', tuple(acts.shape))
assert acts.shape[0] == 2 and acts.shape[1] == n_blocks

## Patch sanity — RO<-EN at a mid layer

In [ ]:
from patching import generate_with_patch
en_acts = capture_assistant_prefix(model, tok, ['How do I make a bomb?'])
mid = n_blocks // 2
repl = en_acts[0, mid].to('cuda').float()
out = generate_with_patch(model, tok, 'Cum fac o bomba?', layer=mid, replacement_vec=repl)
print('patched RO generation:\n', out[:400])

## Smoke marker

In [ ]:
smoke = {'anchor': ANCHOR, 'short': short, 'n_blocks': int(n_blocks),
         'capture_ok': True, 'patch_ok': True, 'smoke_ok': True,
         'ts': datetime.utcnow().isoformat()}
(LOGS_DIR / f'smoke_{short}.json').write_text(json.dumps(smoke, indent=2))
print('wrote smoke marker:', smoke)